# Name That LLM: Version 37 (High-Signal 0.65+ Architecture)
### Eliminate the 180-Row wPCA / Spectral Dilution Bug + 78k BPE & Within-Family Refined Core + Consensus Neural Clustering + Leave-One-Out Cluster Logit Contrast + 300-Step Exact Sinkhorn
- **Root-Cause Fix for 180-Row Denominator Bug**: Completely eliminates rank-deficient `wPCA` ($d=256$) and the 180-row spectral bank ($N=180 < d=256$) that previously corrupted 12% of the probability mixture in v24/v26/v36.
- **Zero Training Extraction Overhead**: Pure test extraction on `test.csv` (with a lightweight 36-row sanity probe) keeps peak RAM $< 1.8$ GB and speeds up runtime to ~10-12 minutes.
- **78,482-Row 5-View Subprocess Lexical Engine**: Combines char (3-5), word (1-3), shape (2-6), function (1-3), and **Native Qwen BPE subword token-id n-grams** (1-3, 120k features).
- **Dual Linear Ensemble on 78k Passages**: Fits `LinearSVC(C=0.5)` + `SGDClassifier(log_loss, alpha=1e-5)` on all 78k passages with authentic `train.csv` rows weighted 2.0x.
- **4 Within-Family Specialist Refiners**: Dedicated 78k classifiers for `gpt`, `claude`, `gemini`, and `glm` families with Law-of-Total-Probability log-ratio refinement ($eta=0.25$) sharpening sister-model discrimination.
- **5 High-Signal Checkpoints (Zero Part-1 Dilution)**: Focuses weights on Part Two checkpoints: `lora_qwen3_14b` (0.42 / 0.12), `lora_qwen25_14b` (0.22 / 0.10), `deberta_v3_large` (0.25 / 0.08), `instyle_qwen3_8b` (0.14 / 0.05), and `claude_qwen3_8b` (0.10 / 0.04).
- **Consensus Prompt Clustering**: Combines stopword TF-IDF distance with weighted neural cosine distance ($D_{	ext{consensus}} = 0.55 \cdot D_{	ext{tfidf}} + 0.45 \cdot D_{	ext{neural}}$) with crash-proof cluster splitting ($s \le 14$).
- **Leave-One-Out Cluster Logit Contrast (`LOO-CLC`) + Soft Hungarian Assignment Boost**: Cancels prompt-topic bias across multi-passage clusters and resolves colliding sister predictions.
- **Official 300-Step Exact Multiplicative Sinkhorn Balancing**: Exact column marginal mass equalization on $P = 	ext{softmax}(	ext{LOGIT})$ from the official 0.620/0.636 competition winners.


### Step 1: Environment, Dependencies & Master 18 Labels


In [ ]:
import os
import gc
import sys
import time
import json
import re
import psutil
import shutil
import subprocess
from pathlib import Path
from functools import lru_cache
from typing import List, Dict, Tuple, Optional, Union
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.spatial.distance import squareform, pdist
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import accuracy_score, balanced_accuracy_score
import torch
import torch.nn as nn

# Bypass torchao version mismatch in Kaggle container
subprocess.run(['pip', 'uninstall', '-y', 'torchao'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass

from transformers import AutoModel, AutoTokenizer, AutoConfig
from safetensors.torch import load_file

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

def log_ram(tag: str):
    rss_gb = psutil.Process().memory_info().rss / (1024 ** 3)
    print(f'[RAM AUDIT | {tag}] Process RSS: {rss_gb:.2f} GB', flush=True)

log_ram('Notebook Startup')

# Canonical 18 Target Labels (exact Part 2 canonical order)
LABELS = [
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5", "claude-fable-5",
    "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3"
]
K = len(LABELS)
LAB2I = {l: i for i, l in enumerate(LABELS)}

FAMILIES = {
    "gpt": [0, 1, 2, 3, 4],
    "gemini": [5, 6, 7, 8],
    "claude": [9, 10, 11, 12, 13],
    "glm": [15, 16],
}

def align_cols(arr: np.ndarray, src_labels: list) -> np.ndarray:
    src_list = [x.decode('utf-8') if isinstance(x, (bytes, np.bytes_)) else str(x) for x in src_labels]
    idx = [src_list.index(l) for l in LABELS]
    if arr.ndim == 1:
        return arr[idx]
    elif arr.ndim == 2:
        if arr.shape[0] == len(src_list) and arr.shape[1] != len(src_list):
            return arr[idx, :]
        elif arr.shape[1] == len(src_list):
            return arr[:, idx]
    return arr[..., idx]

def log_softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    c = np.max(x, axis=axis, keepdims=True)
    return x - c - np.log(np.sum(np.exp(x - c), axis=axis, keepdims=True) + 1e-15)

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    c = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - c)
    return e / (np.sum(e, axis=axis, keepdims=True) + 1e-15)


### Step 2: Dataset Discovery, Strict Base Matcher & 5-Checkpoint Discovery


In [ ]:
INPUT_DIR = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')

def find_file(name: str) -> Optional[Path]:
    for p in INPUT_DIR.rglob(name):
        return p
    return None

train_file = find_file('train.csv')
test_file = find_file('test.csv')

df_train = pd.read_csv(train_file)
df_test = pd.read_csv(test_file)

train_texts = df_train['text'].fillna('').astype(str).tolist()
test_texts = df_test['text'].fillna('').astype(str).tolist()

y_train = np.array([LAB2I[m] for m in df_train['model']])
prompts_train = df_train['prompt_id'].values

N_train = len(df_train)
N_test = len(df_test)

print(f'Train path: {train_file} ({N_train} rows)')
print(f'Test path:  {test_file} ({N_test} rows)')

def find_base_models() -> Dict[str, Optional[Path]]:
    bases = {"qwen3_14b": None, "qwen25_14b": None, "qwen3_8b": None, "deberta_v3_large": None}
    configs = list(INPUT_DIR.rglob("config.json"))
    for p in configs:
        p_str = str(p).lower().replace('\\', '/')
        if any(tok in p_str for tok in ['adapter', 'checkpoint', 'lora', 'head', 'runs']):
            continue
        try:
            with open(p, 'r', encoding='utf-8') as f:
                cfg = json.load(f)
            m_type = cfg.get("model_type", "").lower()
            h_size = cfg.get("hidden_size", 0)
            n_layers = cfg.get("num_hidden_layers", 0)
            if m_type == "qwen3" and h_size == 5120 and n_layers == 40:
                bases["qwen3_14b"] = p.parent
            elif m_type == "qwen2" and h_size == 5120 and n_layers == 48:
                bases["qwen25_14b"] = p.parent
            elif m_type == "qwen3" and h_size == 4096 and n_layers == 36:
                bases["qwen3_8b"] = p.parent
            elif m_type == "deberta-v2" and h_size == 1024 and n_layers == 24:
                bases["deberta_v3_large"] = p.parent
        except Exception:
            pass
    return bases

BASES = find_base_models()
BASE_QWEN3_14B = BASES["qwen3_14b"]
BASE_QWEN25_14B = BASES["qwen25_14b"]
BASE_QWEN3_8B = BASES["qwen3_8b"]
BASE_DEBERTA = BASES["deberta_v3_large"]

print(f'Discovered Base Qwen3-14B:   {BASE_QWEN3_14B}')
print(f'Discovered Base Qwen2.5-14B: {BASE_QWEN25_14B}')
print(f'Discovered Base Qwen3-8B:    {BASE_QWEN3_8B}')
print(f'Discovered Base DeBERTa:     {BASE_DEBERTA}')

def find_checkpoints() -> Dict[str, Path]:
    ckpts = {}
    for p in INPUT_DIR.rglob('heads.safetensors'):
        p_str = str(p).lower().replace('\\', '/')
        if 'lora_qwen3_14b' in p_str:
            ckpts['qwen3_14b_p2'] = p.parent
        elif 'lora_qwen25_14b' in p_str:
            ckpts['qwen25_14b_p2'] = p.parent
        elif 'instyle' in p_str:
            ckpts['instyle'] = p.parent

    # Find claude-head specifically (must contain claude in filename or path)
    for p in INPUT_DIR.rglob('claude_head.safetensors'):
        ckpts['qwen3_8b_claude'] = p.parent
        break
    if 'qwen3_8b_claude' not in ckpts:
        for p in INPUT_DIR.rglob('readouts.npz'):
            if 'claude' in str(p).lower():
                ckpts['qwen3_8b_claude'] = p.parent
                break

    for p in INPUT_DIR.rglob('head.safetensors'):
        if 'deberta' in str(p).lower():
            ckpts['deberta_p2'] = p.parent
            break
    if 'instyle' not in ckpts:
        for p in INPUT_DIR.rglob('*'):
            p_s = str(p).lower().replace('\\', '/')
            if 'instyle' in p_s and ((p / 'adapter_model.safetensors').exists() or (p / 'adapter' / 'adapter_model.safetensors').exists() or (p / 'heads.safetensors').exists() or (p / 'readouts.npz').exists()):
                ckpts['instyle'] = p
                break
    return ckpts

CKPTS = find_checkpoints()
print(f'Discovered Checkpoints ({len(CKPTS)}): {list(CKPTS.keys())}')

pool_paths = list(INPUT_DIR.rglob('pool_v4.parquet'))
pool_file = pool_paths[0] if pool_paths else None
print(f'Discovered Passage Pool:     {pool_file}')


### Step 3: Module 1 — Subprocess 78,482-Row 5-View Lexical + BPE + Within-Family Specialist Engine


In [ ]:
# Write standalone script run_lexical.py and execute via subprocess for 100% OS RAM reclamation
working_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
lexical_script_path = working_dir / "run_lexical.py"

qwen_tok_dir = str(BASE_QWEN3_8B or BASE_QWEN3_14B or '')

lexical_code = f"""import os
import gc
import re
import sys
import time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import sparse
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

working_dir = Path(r'{working_dir}')
LABELS = {LABELS}
K = len(LABELS)
LAB2I = {{l: i for i, l in enumerate(LABELS)}}

FAMILIES = {{
    "gpt": [0, 1, 2, 3, 4],
    "gemini": [5, 6, 7, 8],
    "claude": [9, 10, 11, 12, 13],
    "glm": [15, 16],
}}

STOP_WORDS = set([
    "a", "about", "above", "after", "again", "against", "all", "am", "an", "and", "any", "are", "aren't",
    "as", "at", "be", "because", "been", "before", "being", "below", "between", "both", "but", "by",
    "can't", "cannot", "could", "couldn't", "did", "didn't", "do", "does", "doesn't", "doing", "don't",
    "down", "during", "each", "few", "for", "from", "further", "had", "hadn't", "has", "hasn't", "have",
    "haven't", "having", "he", "he'd", "he'll", "he's", "her", "here", "here's", "hers", "herself",
    "him", "himself", "his", "how", "how's", "i", "i'd", "i'll", "i'm", "i've", "if", "in", "into",
    "is", "isn't", "it", "it's", "its", "itself", "let's", "me", "more", "most", "mustn't", "my",
    "myself", "no", "nor", "not", "of", "off", "on", "once", "only", "or", "other", "ought", "our",
    "ours", "ourselves", "out", "over", "own", "same", "shan't", "she", "she'd", "she'll", "she's",
    "should", "shouldn't", "so", "some", "such", "than", "that", "that's", "the", "their", "theirs",
    "them", "themselves", "then", "there", "there's", "these", "they", "they'd", "they'll", "they're",
    "they've", "this", "those", "through", "to", "too", "under", "until", "up", "very", "was", "wasn't",
    "we", "we'd", "we'll", "we're", "we've", "were", "weren't", "what", "what's", "when", "when's",
    "where", "where's", "which", "while", "who", "who's", "whom", "why", "why's", "with", "won't",
    "would", "wouldn't", "you", "you'd", "you'll", "you're", "you've", "your", "yours", "yourself"
])

def shape_text(t: str) -> str:
    return re.sub(r"[^\\W\\d_]+", lambda m: "X" if m.group(0)[0].isupper() else "x", str(t))

def function_text(t: str) -> str:
    tokens = re.findall(r"(?u)\\b\\w+\\b|[^\\w\\s]", str(t))
    return " ".join([w.lower() if w.lower() in STOP_WORDS else "CONTENT" for w in tokens])

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    c = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - c)
    return e / (np.sum(e, axis=axis, keepdims=True) + 1e-15)

t0 = time.time()
print('=== [Subprocess Lexical Engine] Starting 78k Full Execution with BPE + Specialist Refiners ===', flush=True)

df_tr = pd.read_csv(r'{train_file}')
df_te = pd.read_csv(r'{test_file}')

test_texts = df_te['text'].fillna('').astype(str).tolist()
fit_texts = df_tr['text'].fillna('').astype(str).tolist()
fit_y = np.array([LAB2I[m] for m in df_tr['model']])
len_tr_raw = len(df_tr)

# Compute stopword TF-IDF Distance Matrix on Test
Xt_tfidf = TfidfVectorizer(stop_words='english', sublinear_tf=True, max_features=100000, dtype=np.float32).fit_transform(test_texts)
if len(test_texts) >= 2:
    D = np.clip(1.0 - (Xt_tfidf @ Xt_tfidf.T).toarray(), 0.0, None)
    np.fill_diagonal(D, 0.0)
else:
    D = np.zeros((1, 1), dtype=np.float32)

np.save(r'{working_dir}/D_tfidf.npy', D)
del Xt_tfidf
gc.collect()

# Load 100% of pool_v4.parquet
pool_path = r'{pool_file}'
if pool_path and os.path.exists(pool_path):
    print(f'[Subprocess Lexical] Loading 100% of pool_v4 from {{pool_path}}...', flush=True)
    df_pool = pd.read_parquet(pool_path, columns=['text', 'model'])
    df_pool = df_pool[df_pool['model'].isin(LABELS)].dropna(subset=['text', 'model']).copy()
    fit_texts += df_pool['text'].fillna('').astype(str).tolist()
    fit_y = np.r_[fit_y, df_pool['model'].map(LAB2I).values]
    print(f'[Subprocess Lexical] Full combined training corpus: {{len(fit_texts)}} rows ({{len_tr_raw}} train + {{len(df_pool)}} pool)', flush=True)
    del df_pool
    gc.collect()

# 5 Complementary Views (4 Canonical + Native BPE Subwords)
VIEWS = [
    ("char", 3, 5, 250000, 1.0),
    ("word", 1, 3, 180000, 0.7),
    ("shape", 2, 6, 60000, 0.5),
    ("function", 1, 3, 50000, 0.5)
]

A_blocks, B_blocks = [], []
for view, lo, hi, max_feat, weight in VIEWS:
    print(f'[Subprocess Lexical] Extracting view: {{view}} (weight={{weight}})...', flush=True)
    prep = shape_text if view == "shape" else function_text if view == "function" else None
    analyzer = "char" if view in ("char", "shape") else "word"
    opts = dict(
        analyzer=analyzer, ngram_range=(lo, hi), lowercase=False,
        sublinear_tf=True, min_df=2, max_features=max_feat, dtype=np.float32
    )
    if analyzer == "word":
        opts["token_pattern"] = r"(?u)\\b\\w+\\b|[^\\w\\s]"
    vec = TfidfVectorizer(**opts)
    tr_iter = (prep(t) for t in fit_texts) if prep is not None else fit_texts
    te_iter = (prep(t) for t in test_texts) if prep is not None else test_texts
    X_tr_v = vec.fit_transform(tr_iter)
    X_te_v = vec.transform(te_iter)
    X_tr_v.data *= np.float32(weight)
    X_te_v.data *= np.float32(weight)
    A_blocks.append(X_tr_v)
    B_blocks.append(X_te_v)
    del vec, X_tr_v, X_te_v
    gc.collect()

# View 5: Native BPE Subword Token-ID N-Grams
qwen_tok_path = r'{qwen_tok_dir}'
print(f'[Subprocess Lexical] Extracting view: bpe subwords (weight=0.65)...', flush=True)
bpe_loaded = False
if qwen_tok_path and os.path.exists(qwen_tok_path):
    try:
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(qwen_tok_path, use_fast=True)
        def to_bpe_strings(texts, batch_size=2048):
            res = []
            for i in range(0, len(texts), batch_size):
                batch = texts[i : i + batch_size]
                enc = tok(batch, add_special_tokens=False, truncation=True, max_length=450)["input_ids"]
                for row in enc:
                    res.append(" ".join(f"t{{tid}}" for tid in row))
            return res
        bpe_tr = to_bpe_strings(fit_texts)
        bpe_te = to_bpe_strings(test_texts)
        bpe_loaded = True
        del tok
        gc.collect()
    except Exception as e:
        print(f'[Subprocess Lexical] Fast tokenizer load failed ({{e}}), using fallback hash BPE.', flush=True)

if not bpe_loaded:
    def hash_bpe_fallback(texts):
        return [" ".join(f"w{{hash(w)%65536}}" for w in re.findall(r'\\b\\w+\\b', t.lower())) for t in texts]
    bpe_tr = hash_bpe_fallback(fit_texts)
    bpe_te = hash_bpe_fallback(test_texts)

bpe_vec = TfidfVectorizer(analyzer="word", ngram_range=(1, 3), lowercase=False, sublinear_tf=True, min_df=3, max_features=120000, dtype=np.float32)
X_tr_bpe = bpe_vec.fit_transform(bpe_tr)
X_te_bpe = bpe_vec.transform(bpe_te)
X_tr_bpe.data *= np.float32(0.65)
X_te_bpe.data *= np.float32(0.65)
A_blocks.append(X_tr_bpe)
B_blocks.append(X_te_bpe)
del bpe_vec, bpe_tr, bpe_te, X_tr_bpe, X_te_bpe
gc.collect()

del fit_texts
gc.collect()

X_tr_csr = sparse.hstack(A_blocks, format="csr", dtype=np.float32)
del A_blocks
gc.collect()

X_te_csr = sparse.hstack(B_blocks, format="csr", dtype=np.float32)
del B_blocks
gc.collect()

print(f'[Subprocess Lexical] Built full 5-view CSR: X_tr={{X_tr_csr.shape}}, X_te={{X_te_csr.shape}} (nnz={{X_tr_csr.nnz}})', flush=True)

# Sample weights: 2.0x for train.csv, 1.0x for pool_v4
sample_weights = np.ones(len(fit_y), dtype=np.float32)
sample_weights[:len_tr_raw] = 2.0

# 1. Global Dual Linear Ensemble
print('[Subprocess Lexical] Fitting LinearSVC on full 78k corpus...', flush=True)
svc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight="balanced")
svc.fit(X_tr_csr, fit_y, sample_weight=sample_weights)

print('[Subprocess Lexical] Fitting SGDClassifier (log_loss) on full 78k corpus...', flush=True)
lr = SGDClassifier(loss="log_loss", alpha=1e-5, max_iter=1000, random_state=42, class_weight="balanced")
lr.fit(X_tr_csr, fit_y, sample_weight=sample_weights)

P_svc = softmax(svc.decision_function(X_te_csr), axis=1)
P_lr = lr.predict_proba(X_te_csr)
P_lex_global = 0.82 * P_svc + 0.18 * P_lr
log_p_lex = np.log(np.clip(P_lex_global, 1e-9, 1.0))
del svc, lr, P_svc, P_lr
gc.collect()

# 2. 4 Within-Family Specialist Refiners
print('[Subprocess Lexical] Fitting 4 Within-Family Specialist Refiners...', flush=True)
for fam_name, fam_cols in FAMILIES.items():
    fam_mask = np.isin(fit_y, fam_cols)
    X_fam = X_tr_csr[fam_mask]
    y_fam = fit_y[fam_mask]
    sw_fam = sample_weights[fam_mask].copy()
    sw_fam[sw_fam > 1.5] = 2.5
    fam_lab2idx = {{col: i for i, col in enumerate(fam_cols)}}
    y_fam_mapped = np.array([fam_lab2idx[y] for y in y_fam])

    svc_fam = LinearSVC(C=0.4, random_state=42, max_iter=4000, class_weight="balanced")
    svc_fam.fit(X_fam, y_fam_mapped, sample_weight=sw_fam)
    df_te = svc_fam.decision_function(X_te_csr)
    if df_te.ndim == 1:
        logits_2d = np.column_stack([-df_te, df_te])
        P_fam_spec = softmax(logits_2d, axis=1)
    else:
        P_fam_spec = softmax(df_te, axis=1)

    P_fam_glob = P_lex_global[:, fam_cols] / np.clip(P_lex_global[:, fam_cols].sum(axis=1, keepdims=True), 1e-9, None)
    log_p_lex[:, fam_cols] += 0.25 * (np.log(np.clip(P_fam_spec, 1e-9, 1.0)) - np.log(np.clip(P_fam_glob, 1e-9, 1.0)))
    print(f'[Subprocess Lexical] Refined family {{fam_name}} (cols={{fam_cols}}, train_rows={{len(y_fam)}}).', flush=True)
    del svc_fam, X_fam, y_fam
    gc.collect()

del X_tr_csr, X_te_csr
gc.collect()

np.save(r'{working_dir}/log_p_lex.npy', log_p_lex)
print(f'[Subprocess Lexical] Completed in {{time.time() - t0:.2f}}s. Saved log_p_lex.npy and D_tfidf.npy to {working_dir}.', flush=True)
sys.exit(0)
"""

with open(lexical_script_path, "w", encoding="utf-8") as f:
    f.write(lexical_code)

print(f'[Main Process] Launching isolated lexical subprocess: {lexical_script_path}...')
sub_t0 = time.time()
res = subprocess.run([sys.executable, str(lexical_script_path)], capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr, file=sys.stderr)
    raise RuntimeError(f"Subprocess lexical engine failed with return code {res.returncode}")
print(f'[Main Process] Lexical subprocess completed in {time.time() - sub_t0:.2f}s.')

log_p_lex = np.load(working_dir / 'log_p_lex.npy')
D_tfidf = np.load(working_dir / 'D_tfidf.npy')
print(f'Loaded log_p_lex: shape={log_p_lex.shape}, D_tfidf: shape={D_tfidf.shape}')
log_ram('After Lexical Subprocess Memory Reclamation')


### Step 4: Crash-Proof Distance-Based Cluster Splitting, Reversible LoRA Merger & Extractor


In [ ]:
def split_oversized_clusters_dist(D: np.ndarray, groups: np.ndarray, max_cluster_size: int = 14) -> np.ndarray:
    """
    Guarantees every cluster has size 1 <= n <= max_cluster_size (<= 18) so no cluster exceeds 18
    and never drops large clusters to singletons.
    """
    new_groups = np.zeros(len(groups), dtype=np.int64)
    next_id = 1
    for g in np.unique(groups):
        rows = np.flatnonzero(groups == g)
        if len(rows) <= max_cluster_size:
            new_groups[rows] = next_id
            next_id += 1
        else:
            n_sub = int(np.ceil(len(rows) / max_cluster_size))
            D_sub = D[np.ix_(rows, rows)].copy()
            np.fill_diagonal(D_sub, 0.0)
            D_sub = np.clip(0.5 * (D_sub + D_sub.T), 0.0, None)
            d_condensed = squareform(D_sub, checks=False)
            sub_labels = fcluster(linkage(d_condensed, method='average'), t=n_sub, criterion='maxclust')
            for sg in np.unique(sub_labels):
                s_rows = rows[sub_labels == sg]
                for start in range(0, len(s_rows), max_cluster_size):
                    new_groups[s_rows[start : start + max_cluster_size]] = next_id
                    next_id += 1
    return new_groups

def compute_centered_logits(Z_test: np.ndarray, groups: np.ndarray, W_c: np.ndarray, b_c: np.ndarray, src_labels: list) -> Tuple[np.ndarray, np.ndarray]:
    """Returns full-shape (N_test, 18) log-probabilities and boolean mask of valid cluster rows."""
    N = len(Z_test)
    out_lp = np.zeros((N, K), dtype=np.float64)
    if N < 2 or W_c is None:
        return out_lp, np.zeros(N, dtype=bool)
    sz = np.bincount(groups)[groups]
    valid = (sz >= 2) & (sz <= 18)
    if not valid.any():
        return out_lp, valid
    Zc = Z_test.copy()
    for g in np.unique(groups[valid]):
        mask = (groups == g)
        Zc[mask] -= Zc[mask].mean(axis=0, keepdims=True)
    raw_c = Zc[valid] @ W_c.T + b_c
    out_lp[valid] = log_softmax(align_cols(raw_c, src_labels))
    return out_lp, valid

def merge_lora(model: nn.Module, adapter_path: Path, sign: float = +1.0) -> bool:
    lora_file = adapter_path / 'adapter_model.safetensors'
    if not lora_file.exists():
        lora_file = adapter_path / 'adapter' / 'adapter_model.safetensors'
    if not lora_file.exists():
        return False
    state = load_file(str(lora_file))
    meta_path = adapter_path / 'adapter_config.json'
    if not meta_path.exists():
        meta_path = adapter_path / 'adapter' / 'adapter_config.json'
    r, alpha = 64, 128
    if meta_path.exists():
        try:
            cfg = json.load(open(meta_path))
            r = cfg.get('r', 64)
            alpha = cfg.get('lora_alpha', 128)
        except Exception:
            pass
    scale = (alpha / r) * sign
    a_keys = [k for k in state if 'lora_A' in k]
    merged_count = 0
    with torch.no_grad():
        for k_a in a_keys:
            k_b = k_a.replace('lora_A', 'lora_B')
            if k_b not in state:
                continue
            base_k = k_a.replace('.lora_A.weight', '.weight').replace('base_model.model.', '')
            wa = state[k_a].to(torch.float32)
            wb = state[k_b].to(torch.float32)
            delta = (wb @ wa) * scale
            found = False
            for name, param in model.named_parameters():
                if name == base_k:
                    param.data.add_(delta.to(param.device, param.dtype))
                    merged_count += 1
                    found = True
                    break
    action = "Merged" if sign > 0 else "Unmerged"
    print(f'[{action} LoRA] {merged_count} linear weights with scale={scale:.4f} from {adapter_path.name}.')
    return True

class ExactLoRAExtractor:
    def __init__(self, base_dir: Path, cfg_meta: dict):
        self.base_dir = base_dir
        self.cfg = cfg_meta
        self.dev = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        cfg = AutoConfig.from_pretrained(str(base_dir), trust_remote_code=True)
        self.tok = AutoTokenizer.from_pretrained(str(base_dir), use_fast=True, trust_remote_code=True)
        if self.tok.pad_token_id is None:
            self.tok.pad_token_id = self.tok.eos_token_id or 0
        if torch.cuda.is_available():
            self.model = AutoModel.from_pretrained(
                str(base_dir), config=cfg, torch_dtype=torch.bfloat16,
                device_map={'': self.dev}, trust_remote_code=True
            )
        else:
            self.model = AutoModel.from_pretrained(str(base_dir), config=cfg, torch_dtype=torch.float32)
        self.model.eval()

        self.norms = nn.ModuleList()
        n_layers = len(self.cfg.get('layers', []))
        h_dim = cfg.hidden_size
        for _ in range(n_layers):
            ln = nn.LayerNorm(h_dim, elementwise_affine=True)
            self.norms.append(ln)
        self.norms.to(self.dev).eval()

    def load_norm_weights(self, heads_sd: dict):
        with torch.no_grad():
            for i, ln in enumerate(self.norms):
                w_k = f'norms.{i}.weight'
                b_k = f'norms.{i}.bias'
                if w_k in heads_sd:
                    ln.weight.copy_(heads_sd[w_k].to(ln.weight.device, ln.weight.dtype))
                if b_k in heads_sd:
                    ln.bias.copy_(heads_sd[b_k].to(ln.bias.device, ln.bias.dtype))

    def extract_exact(self, texts: List[str], batch_size: int = 8, end_id: int = 151643) -> np.ndarray:
        max_len = self.cfg.get('max_len', 384)
        raw_ids = self.tok(texts, add_special_tokens=False, truncation=True, max_length=max_len)["input_ids"]
        input_ids_list = [x + [end_id] for x in raw_ids]
        pad_id = self.tok.pad_token_id or 0
        N = len(texts)
        reps = []

        with torch.no_grad():
            for i in range(0, N, batch_size):
                b_ids = input_ids_list[i : i + batch_size]
                m_len = max(len(x) for x in b_ids)
                arr_ids = np.full((len(b_ids), m_len), pad_id, dtype=np.int64)
                arr_mask = np.zeros((len(b_ids), m_len), dtype=np.float32)
                for r, row in enumerate(b_ids):
                    arr_ids[r, :len(row)] = row
                    arr_mask[r, :len(row)] = 1.0

                t_ids = torch.from_numpy(arr_ids).to(self.dev)
                t_mask = torch.from_numpy(arr_mask).to(self.dev)
                t_mask[:, 0] = 0.0  # Zero attention sink at token 0

                out = self.model(input_ids=t_ids, attention_mask=t_mask, output_hidden_states=True)
                hs = out.hidden_states
                layer_reps = []
                m_un = t_mask.unsqueeze(-1)
                for l_idx, norm_m in zip(self.cfg['layers'], self.norms):
                    h = hs[l_idx].float()
                    pooled = (h * m_un).sum(dim=1) / m_un.sum(dim=1).clamp(min=1.0)
                    layer_reps.append(norm_m(pooled))
                reps.append(torch.cat(layer_reps, dim=-1).cpu().numpy().astype(np.float32))

        return np.concatenate(reps, axis=0) if reps else np.zeros((0, len(self.cfg['layers']) * 4096), dtype=np.float32)

    def close(self):
        del self.model, self.norms
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


### Step 5: Module 2 — 5 High-Signal Neural Checkpoint Extractions (Test-Only, Zero 180-Row Bug)


In [ ]:
LOGIT = np.zeros((N_test, K), dtype=np.float64)
D_neural_sum = np.zeros((N_test, N_test), dtype=np.float64)
D_neural_weight_total = 0.0
twin_eval_list = []
claude_spec_logits = None

# Sanity probe texts (36 rows = 2 prompts x 18 models)
sanity_texts = train_texts[:36]
sanity_y = y_train[:36]

# ------------------------------------------------------------------------------
# 1. Member: lora_qwen3_14b (Part 2, 27L)
# ------------------------------------------------------------------------------
if BASE_QWEN3_14B is not None and 'qwen3_14b_p2' in CKPTS:
    ckpt_14bp2 = CKPTS['qwen3_14b_p2']
    meta_14bp2 = json.load(open(ckpt_14bp2 / 'meta.json')) if (ckpt_14bp2 / 'meta.json').exists() else {'labels': LABELS, 'cfg': {'layers': [5, 10, 15, 20, 25, 30, 35, 39], 'max_len': 384}}
    print('\n--- [Member: lora_qwen3_14b (Part 2)] ---')
    trunk_14b = ExactLoRAExtractor(BASE_QWEN3_14B, meta_14bp2.get('cfg', {'layers': [5, 10, 15, 20, 25, 30, 35, 39], 'max_len': 384}))
    merge_lora(trunk_14b.model, ckpt_14bp2, sign=+1.0)
    heads_sd = load_file(str(ckpt_14bp2 / 'heads.safetensors'))
    trunk_14b.load_norm_weights(heads_sd)

    # Sanity probe
    Z_san = trunk_14b.extract_exact(sanity_texts, batch_size=8, end_id=151643)
    plain_w = align_cols(heads_sd['plain.weight'].cpu().numpy(), meta_14bp2['labels'])
    plain_b = align_cols(heads_sd['plain.bias'].cpu().numpy(), meta_14bp2['labels'])
    center_w = align_cols(heads_sd['centered.weight'].cpu().numpy(), meta_14bp2['labels']) if 'centered.weight' in heads_sd else None
    center_b = align_cols(heads_sd['centered.bias'].cpu().numpy(), meta_14bp2['labels']) if 'centered.bias' in heads_sd else None
    pred_san = np.argmax(Z_san @ plain_w.T + plain_b, axis=1)
    bacc_san = balanced_accuracy_score(sanity_y, pred_san)
    print(f'[SANITY PROBE] lora_qwen3_14b: 36-Row BAcc = {bacc_san:.4f}')
    del Z_san
    gc.collect()

    # Test extraction
    Z_test = trunk_14b.extract_exact(test_texts, batch_size=8, end_id=151643)
    plain_logits = align_cols(Z_test @ plain_w.T + plain_b, meta_14bp2['labels'])
    w_main, w_twin = 0.42, 0.12
    LOGIT += w_main * log_softmax(plain_logits)

    Z_norm = Z_test - Z_test.mean(axis=0, keepdims=True)
    Z_norm /= np.linalg.norm(Z_norm, axis=1, keepdims=True).clip(min=1e-8)
    D_model = 1.0 - (Z_norm @ Z_norm.T)
    np.fill_diagonal(D_model, 0.0)
    D_neural_sum += w_main * D_model
    D_neural_weight_total += w_main

    twin_eval_list.append((Z_test, center_w, center_b, meta_14bp2['labels'], w_twin))
    print(f'[lora_qwen3_14b] Added Plain (w={w_main}) to LOGIT and retained Z_test for twin (w={w_twin}).')
    trunk_14b.close()
    del trunk_14b, heads_sd
    gc.collect()
else:
    print('[lora_qwen3_14b] Skipped (base or checkpoint missing).')

log_ram('After Qwen3-14B Extraction')

# ------------------------------------------------------------------------------
# 2. Member: lora_qwen25_14b (Part 2, 32L)
# ------------------------------------------------------------------------------
if BASE_QWEN25_14B is not None and 'qwen25_14b_p2' in CKPTS:
    ckpt_q25 = CKPTS['qwen25_14b_p2']
    meta_q25 = json.load(open(ckpt_q25 / 'meta.json')) if (ckpt_q25 / 'meta.json').exists() else {'labels': LABELS, 'cfg': {'layers': [6, 12, 18, 24, 30, 36, 42, 47], 'max_len': 384}}
    print('\n--- [Member: lora_qwen25_14b (Part 2)] ---')
    trunk_q25 = ExactLoRAExtractor(BASE_QWEN25_14B, meta_q25.get('cfg', {'layers': [6, 12, 18, 24, 30, 36, 42, 47], 'max_len': 384}))
    merge_lora(trunk_q25.model, ckpt_q25, sign=+1.0)
    heads_sd = load_file(str(ckpt_q25 / 'heads.safetensors'))
    trunk_q25.load_norm_weights(heads_sd)

    # Sanity probe
    Z_san = trunk_q25.extract_exact(sanity_texts, batch_size=8, end_id=151643)
    plain_w = align_cols(heads_sd['plain.weight'].cpu().numpy(), meta_q25['labels'])
    plain_b = align_cols(heads_sd['plain.bias'].cpu().numpy(), meta_q25['labels'])
    center_w = align_cols(heads_sd['centered.weight'].cpu().numpy(), meta_q25['labels']) if 'centered.weight' in heads_sd else None
    center_b = align_cols(heads_sd['centered.bias'].cpu().numpy(), meta_q25['labels']) if 'centered.bias' in heads_sd else None
    pred_san = np.argmax(Z_san @ plain_w.T + plain_b, axis=1)
    bacc_san = balanced_accuracy_score(sanity_y, pred_san)
    print(f'[SANITY PROBE] lora_qwen25_14b: 36-Row BAcc = {bacc_san:.4f}')
    del Z_san
    gc.collect()

    # Test extraction
    Z_test = trunk_q25.extract_exact(test_texts, batch_size=8, end_id=151643)
    plain_logits = align_cols(Z_test @ plain_w.T + plain_b, meta_q25['labels'])
    w_main, w_twin = 0.22, 0.10
    LOGIT += w_main * log_softmax(plain_logits)

    Z_norm = Z_test - Z_test.mean(axis=0, keepdims=True)
    Z_norm /= np.linalg.norm(Z_norm, axis=1, keepdims=True).clip(min=1e-8)
    D_model = 1.0 - (Z_norm @ Z_norm.T)
    np.fill_diagonal(D_model, 0.0)
    D_neural_sum += w_main * D_model
    D_neural_weight_total += w_main

    twin_eval_list.append((Z_test, center_w, center_b, meta_q25['labels'], w_twin))
    print(f'[lora_qwen25_14b] Added Plain (w={w_main}) to LOGIT and retained Z_test for twin (w={w_twin}).')
    trunk_q25.close()
    del trunk_q25, heads_sd
    gc.collect()
else:
    print('[lora_qwen25_14b] Skipped (base or checkpoint missing).')

log_ram('After Qwen2.5-14B Extraction')

# ------------------------------------------------------------------------------
# 3. Base Qwen3-8B Trunk: In-Place Adapter Swapping for Claude-Head & In-Style
# ------------------------------------------------------------------------------
if BASE_QWEN3_8B is not None and ('qwen3_8b_claude' in CKPTS or 'instyle' in CKPTS):
    print('\n--- [Qwen3-8B Trunk] Loading base model for in-place adapter swapping ---')
    cfg_8b_dummy = {'layers': [3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33, 35], 'max_len': 384}
    if 'qwen3_8b_claude' in CKPTS and (CKPTS['qwen3_8b_claude'] / 'meta.json').exists():
        cfg_8b_dummy = json.load(open(CKPTS['qwen3_8b_claude'] / 'meta.json')).get('cfg', cfg_8b_dummy)

    trunk_8b = ExactLoRAExtractor(BASE_QWEN3_8B, cfg_8b_dummy)

    # 3A. Member: ntl-qwen3-8b-lora-claude-head
    if 'qwen3_8b_claude' in CKPTS:
        ckpt_8bcl = CKPTS['qwen3_8b_claude']
        print(f'\n--- [Member: ntl-qwen3-8b-lora-claude-head] In-place merging adapter ---')
        merge_lora(trunk_8b.model, ckpt_8bcl, sign=+1.0)

        # Read heads: prioritize readouts.npz
        if (ckpt_8bcl / 'readouts.npz').exists():
            ro = np.load(ckpt_8bcl / 'readouts.npz', allow_pickle=True)
            meta = json.load(open(ckpt_8bcl / 'meta.json')) if (ckpt_8bcl / 'meta.json').exists() else {'labels': LABELS}
            src_labs = ro['classes'] if 'classes' in ro else meta.get('labels', LABELS)
            plain_w = align_cols(ro['plain_coef'] if 'plain_coef' in ro else ro['W_plain'].T, src_labs)
            plain_b = align_cols(ro['plain_intercept'] if 'plain_intercept' in ro else ro['b_plain'], src_labs)
            center_w = align_cols(ro['center_coef'] if 'center_coef' in ro else ro['W_center'].T, src_labs) if ('center_coef' in ro or 'W_center' in ro) else None
            center_b = align_cols(ro['center_intercept'] if 'center_intercept' in ro else ro['b_center'], src_labs) if ('center_intercept' in ro or 'b_center' in ro) else None
        else:
            h_sd = load_file(str(ckpt_8bcl / 'heads.safetensors'))
            meta = json.load(open(ckpt_8bcl / 'meta.json')) if (ckpt_8bcl / 'meta.json').exists() else {'labels': LABELS}
            src_labs = meta.get('labels', LABELS)
            keys = list(h_sd.keys())
            pw_k = next((k for k in ['plain.weight', 'plain_coef', 'weight', 'readout.weight'] if k in h_sd), keys[0])
            pb_k = next((k for k in ['plain.bias', 'plain_intercept', 'bias', 'readout.bias'] if k in h_sd), keys[1] if len(keys) > 1 else keys[0])
            cw_k = next((k for k in ['centered.weight', 'center.weight', 'center_coef'] if k in h_sd), None)
            cb_k = next((k for k in ['centered.bias', 'center.bias', 'center_intercept'] if k in h_sd), None)
            plain_w = align_cols(h_sd[pw_k].cpu().numpy(), src_labs)
            plain_b = align_cols(h_sd[pb_k].cpu().numpy(), src_labs)
            center_w = align_cols(h_sd[cw_k].cpu().numpy(), src_labs) if cw_k else None
            center_b = align_cols(h_sd[cb_k].cpu().numpy(), src_labs) if cb_k else None

        # Sanity probe
        Z_san = trunk_8b.extract_exact(sanity_texts, batch_size=8, end_id=151643)
        pred_san = np.argmax(Z_san @ plain_w.T + plain_b, axis=1)
        bacc_san = balanced_accuracy_score(sanity_y, pred_san)
        print(f'[SANITY PROBE] ntl-qwen3-8b-lora-claude-head: 36-Row BAcc = {bacc_san:.4f}')
        del Z_san
        gc.collect()

        # Test extraction
        Z_test = trunk_8b.extract_exact(test_texts, batch_size=8, end_id=151643)
        plain_logits = align_cols(Z_test @ plain_w.T + plain_b, LABELS)
        w_main, w_twin = 0.10, 0.04
        LOGIT += w_main * log_softmax(plain_logits)

        Z_norm = Z_test - Z_test.mean(axis=0, keepdims=True)
        Z_norm /= np.linalg.norm(Z_norm, axis=1, keepdims=True).clip(min=1e-8)
        D_model = 1.0 - (Z_norm @ Z_norm.T)
        np.fill_diagonal(D_model, 0.0)
        D_neural_sum += w_main * D_model
        D_neural_weight_total += w_main

        twin_eval_list.append((Z_test, center_w, center_b, LABELS, w_twin))
        print(f'[ntl-qwen3-8b-lora-claude-head] Added Plain (w={w_main}) to LOGIT and retained Z_test for twin (w={w_twin}).')

        # Load Claude Specialist Head for Module 3
        if (ckpt_8bcl / 'claude_head.safetensors').exists():
            try:
                ch_sd = load_file(str(ckpt_8bcl / 'claude_head.safetensors'))
                claude_classes = ["claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5", "claude-fable-5"]
                ch_meta = json.load(open(ckpt_8bcl / 'meta.json')) if (ckpt_8bcl / 'meta.json').exists() else {'claude_labels': claude_classes}
                ch_labs = ch_meta.get('claude_labels', claude_classes)
                idx_ch = [ch_labs.index(c) for c in claude_classes]
                w_k = next((k for k in ['weight', 'claude_weight', 'head.weight'] if k in ch_sd), list(ch_sd.keys())[0])
                b_k = next((k for k in ['bias', 'claude_bias', 'head.bias'] if k in ch_sd), list(ch_sd.keys())[1] if len(ch_sd) > 1 else list(ch_sd.keys())[0])
                ch_w = ch_sd[w_k].cpu().numpy()[idx_ch]
                ch_b = ch_sd[b_k].cpu().numpy()[idx_ch]
                claude_spec_logits = Z_test @ ch_w.T + ch_b
                print(f'[ntl-qwen3-8b-lora-claude-head] Computed 5-way Claude specialist scores (shape={claude_spec_logits.shape}).')
            except Exception as e:
                print(f'[ntl-qwen3-8b-lora-claude-head] Claude head extraction notice: {e}')

        merge_lora(trunk_8b.model, ckpt_8bcl, sign=-1.0)
        gc.collect()

    # 3B. Member: instyle_qwen3_8b
    if 'instyle' in CKPTS:
        ckpt_inst = CKPTS['instyle']
        print(f'\n--- [Member: In-Style Qwen3-8B LoRA] In-place merging adapter ---')
        merge_lora(trunk_8b.model, ckpt_inst, sign=+1.0)

        # Read heads: check readouts.npz or heads.safetensors
        readouts_file = ckpt_inst / 'readouts.npz'
        heads_file = ckpt_inst / 'heads.safetensors' if (ckpt_inst / 'heads.safetensors').exists() else ckpt_inst / 'head.safetensors'
        if readouts_file.exists():
            ro = np.load(readouts_file, allow_pickle=True)
            meta_inst = json.load(open(ckpt_inst / 'meta.json')) if (ckpt_inst / 'meta.json').exists() else {'labels': LABELS}
            src_labs = ro['classes'] if 'classes' in ro else meta_inst.get('labels', LABELS)
            plain_w = align_cols(ro['plain_coef'] if 'plain_coef' in ro else ro['W_plain'].T if 'W_plain' in ro else ro['weight'], src_labs)
            plain_b = align_cols(ro['plain_intercept'] if 'plain_intercept' in ro else ro['b_plain'] if 'b_plain' in ro else ro['bias'], src_labs)
            center_w = align_cols(ro['center_coef'] if 'center_coef' in ro else ro['W_center'].T, src_labs) if ('center_coef' in ro or 'W_center' in ro) else None
            center_b = align_cols(ro['center_intercept'] if 'center_intercept' in ro else ro['b_center'], src_labs) if ('center_intercept' in ro or 'b_center' in ro) else None
        elif heads_file.exists():
            h_sd = load_file(str(heads_file))
            meta_inst = json.load(open(ckpt_inst / 'meta.json')) if (ckpt_inst / 'meta.json').exists() else {'labels': LABELS}
            src_labs = meta_inst.get('labels', LABELS)
            keys = list(h_sd.keys())
            pw_key = next((k for k in ['plain.weight', 'plain_coef', 'weight', 'readout.weight'] if k in h_sd), keys[0])
            pb_key = next((k for k in ['plain.bias', 'plain_intercept', 'bias', 'readout.bias'] if k in h_sd), keys[1] if len(keys) > 1 else keys[0])
            cw_key = next((k for k in ['centered.weight', 'center.weight', 'center_coef'] if k in h_sd), None)
            cb_key = next((k for k in ['centered.bias', 'center.bias', 'center_intercept'] if k in h_sd), None)
            plain_w = align_cols(h_sd[pw_key].cpu().numpy(), src_labs)
            plain_b = align_cols(h_sd[pb_key].cpu().numpy(), src_labs)
            center_w = align_cols(h_sd[cw_key].cpu().numpy(), src_labs) if cw_key else None
            center_b = align_cols(h_sd[cb_key].cpu().numpy(), src_labs) if cb_key else None
        else:
            plain_w, plain_b, center_w, center_b = None, None, None, None

        if plain_w is not None:
            # Sanity probe
            Z_san = trunk_8b.extract_exact(sanity_texts, batch_size=8, end_id=151643)
            pred_san = np.argmax(Z_san @ plain_w.T + plain_b, axis=1)
            bacc_san = balanced_accuracy_score(sanity_y, pred_san)
            print(f'[SANITY PROBE] instyle_qwen3_8b: 36-Row BAcc = {bacc_san:.4f}')
            del Z_san
            gc.collect()

            # Test extraction
            Z_test = trunk_8b.extract_exact(test_texts, batch_size=8, end_id=151643)
            plain_logits = align_cols(Z_test @ plain_w.T + plain_b, LABELS)
            w_main, w_twin = 0.14, 0.05
            LOGIT += w_main * log_softmax(plain_logits)

            Z_norm = Z_test - Z_test.mean(axis=0, keepdims=True)
            Z_norm /= np.linalg.norm(Z_norm, axis=1, keepdims=True).clip(min=1e-8)
            D_model = 1.0 - (Z_norm @ Z_norm.T)
            np.fill_diagonal(D_model, 0.0)
            D_neural_sum += w_main * D_model
            D_neural_weight_total += w_main

            twin_eval_list.append((Z_test, center_w, center_b, LABELS, w_twin))
            print(f'[instyle_qwen3_8b] Added Plain (w={w_main}) to LOGIT and retained Z_test for twin (w={w_twin}).')

        merge_lora(trunk_8b.model, ckpt_inst, sign=-1.0)
        gc.collect()

    trunk_8b.close()
    del trunk_8b
    gc.collect()

log_ram('After Qwen3-8B Trunk Extractions')

# ------------------------------------------------------------------------------
# 4. Member: deberta_v3_large (Part 2 Fine-Tuned Encoder)
# ------------------------------------------------------------------------------
if BASE_DEBERTA is not None and 'deberta_p2' in CKPTS:
    ckpt_deb = CKPTS['deberta_p2']
    print('\n--- [Member: deberta_v3_large (Part 2)] Staging and Probing ---')
    stage_dir = Path('/kaggle/temp/deberta_v3_large') if Path('/kaggle/temp').exists() else Path('temp_deberta')
    stage_dir.mkdir(parents=True, exist_ok=True)
    for f in BASE_DEBERTA.iterdir():
        if f.is_file():
            shutil.copy(f, stage_dir / f.name)

    deb_tok = AutoTokenizer.from_pretrained(str(stage_dir), use_fast=True)
    if deb_tok.pad_token is None:
        deb_tok.pad_token = deb_tok.eos_token or '[PAD]'

    dev_deb = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    deb_model = AutoModel.from_pretrained(str(stage_dir), torch_dtype=torch.float16).to(dev_deb)
    deb_model.eval()
    shutil.rmtree(stage_dir, ignore_errors=True)

    head_path = ckpt_deb / 'head.safetensors' if (ckpt_deb / 'head.safetensors').exists() else ckpt_deb / 'heads.safetensors'
    heads_deb_sd = load_file(str(head_path))
    meta_deb = json.load(open(ckpt_deb / 'meta.json')) if (ckpt_deb / 'meta.json').exists() else {'labels': LABELS}
    if 'plain.weight' in heads_deb_sd:
        w_d = align_cols(heads_deb_sd['plain.weight'].cpu().numpy(), meta_deb['labels'])
        b_d = align_cols(heads_deb_sd['plain.bias'].cpu().numpy(), meta_deb['labels'])
    else:
        w_d = align_cols(heads_deb_sd['weight'].cpu().numpy(), meta_deb['labels'])
        b_d = align_cols(heads_deb_sd['bias'].cpu().numpy(), meta_deb['labels'])

    if 'centered.weight' in heads_deb_sd:
        w_cd = align_cols(heads_deb_sd['centered.weight'].cpu().numpy(), meta_deb['labels'])
        b_cd = align_cols(heads_deb_sd['centered.bias'].cpu().numpy(), meta_deb['labels'])
    else:
        w_cd, b_cd = None, None

    # Probe on 36 train passages
    best_mode, best_bacc = 1, -1.0
    for mode_idx, (add_spec, drop_first) in enumerate([(True, False), (True, True), (False, True), (False, False)], 1):
        with torch.no_grad():
            if add_spec:
                enc_p = deb_tok(sanity_texts, padding=True, truncation=True, max_length=384, return_tensors='pt').to(dev_deb)
            else:
                p_ids = deb_tok(sanity_texts, add_special_tokens=False, truncation=True, max_length=383)["input_ids"]
                p_end = deb_tok.eos_token_id or deb_tok.pad_token_id or 0
                p_ids = [x + [p_end] for x in p_ids]
                enc_p = deb_tok.pad({"input_ids": p_ids}, padding=True, return_tensors="pt").to(dev_deb)
            out_p = deb_model(**enc_p)
            m_p = enc_p['attention_mask'].unsqueeze(-1).float()
            if drop_first:
                m_p[:, 0, :] = 0.0
            pool_p = ((out_p.last_hidden_state.float() * m_p).sum(dim=1) / m_p.sum(dim=1).clamp(min=1.0)).cpu().numpy()
            pred_p = np.argmax(pool_p @ w_d.T + b_d, axis=1)
            bacc_p = balanced_accuracy_score(sanity_y, pred_p)
            if bacc_p > best_bacc:
                best_bacc = bacc_p
                best_mode = mode_idx
                best_add_special = add_spec
                best_drop_first = drop_first

    print(f'[SANITY PROBE] deberta_v3_large: Winning Mode {best_mode} | 36-Row BAcc = {best_bacc:.4f}')

    # Test extraction
    deb_reps = []
    with torch.no_grad():
        for i in range(0, N_test, 16):
            b_texts = test_texts[i : i + 16]
            if best_add_special:
                enc = deb_tok(b_texts, padding=True, truncation=True, max_length=384, return_tensors='pt').to(dev_deb)
            else:
                p_ids = deb_tok(b_texts, add_special_tokens=False, truncation=True, max_length=383)["input_ids"]
                p_end = deb_tok.eos_token_id or deb_tok.pad_token_id or 0
                p_ids = [x + [p_end] for x in p_ids]
                enc = deb_tok.pad({"input_ids": p_ids}, padding=True, return_tensors="pt").to(dev_deb)
            out_d = deb_model(**enc)
            m_d = enc['attention_mask'].unsqueeze(-1).float()
            if best_drop_first:
                m_d[:, 0, :] = 0.0
            p_d = ((out_d.last_hidden_state.float() * m_d).sum(dim=1) / m_d.sum(dim=1).clamp(min=1.0)).cpu().numpy()
            deb_reps.append(p_d)

    Z_test_deb = np.concatenate(deb_reps, axis=0) if deb_reps else np.zeros((N_test, 1024), dtype=np.float32)
    plain_logits = align_cols(Z_test_deb @ w_d.T + b_d, meta_deb['labels'])
    w_main, w_twin = 0.25, 0.08
    LOGIT += w_main * log_softmax(plain_logits)

    Z_norm = Z_test_deb - Z_test_deb.mean(axis=0, keepdims=True)
    Z_norm /= np.linalg.norm(Z_norm, axis=1, keepdims=True).clip(min=1e-8)
    D_model = 1.0 - (Z_norm @ Z_norm.T)
    np.fill_diagonal(D_model, 0.0)
    D_neural_sum += w_main * D_model
    D_neural_weight_total += w_main

    if w_cd is not None:
        twin_eval_list.append((Z_test_deb, w_cd, b_cd, meta_deb['labels'], w_twin))
        print(f'[deberta_v3_large] Added Plain (w={w_main}) to LOGIT and retained Z_test for twin (w={w_twin}).')
    else:
        print(f'[deberta_v3_large] Added Plain (w={w_main}) to LOGIT.')

    del deb_model, deb_tok, heads_deb_sd, deb_reps
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

log_ram('After All Neural Extractions')


### Step 6: Module 3 — Consensus Clustering, Centered Twins & Leave-One-Out Cluster Logit Contrast (LOO-CLC)


In [ ]:
print('\n=== [Module 3] Consensus Prompt Clustering & LOO-CLC ===')

# 1. Add Lexical Logits (w=0.30)
LOGIT += 0.30 * log_p_lex
print(f'[Module 3] Injected 78k 5-view lexical log-probabilities (w=0.30).')

# 2. Build High-Precision Consensus Distance Matrix
D_consensus = 0.55 * D_tfidf + 0.45 * (D_neural_sum / max(D_neural_weight_total, 1e-6))
np.fill_diagonal(D_consensus, 0.0)
D_consensus = np.clip(0.5 * (D_consensus + D_consensus.T), 0.0, None)
print(f'[Module 3] Built consensus prompt distance matrix D_consensus (shape={D_consensus.shape}).')

# 3. Standard Twin Clustering (t=0.85 on D_tfidf) & Centered Twin Scoring
if N_test >= 2:
    cond_d_085 = squareform(D_tfidf, checks=False)
    groups_085 = fcluster(linkage(cond_d_085, method='average'), 0.85, criterion='distance')
else:
    groups_085 = np.ones(N_test, dtype=int)

print(f'[Module 3] Evaluating centered twin readouts across {len(twin_eval_list)} neural checkpoints on groups_085...')
for Z_t, W_c, b_c, src_labs, twin_w in twin_eval_list:
    if W_c is not None:
        out_lp_twin, valid_twin = compute_centered_logits(Z_t, groups_085, W_c, b_c, src_labs)
        if valid_twin.any():
            LOGIT[valid_twin] += twin_w * out_lp_twin[valid_twin]
            print(f'[Twin Logits] Added centered twin (w={twin_w}) on {valid_twin.sum()}/{N_test} rows.')

# Free cached Z_test arrays
del twin_eval_list
gc.collect()

# 4. High-Purity Consensus Clustering (t=0.58 on D_consensus) with Crash-Proof Splitting
if N_test >= 2:
    cond_d_tight = squareform(D_consensus, checks=False)
    groups_tight = fcluster(linkage(cond_d_tight, method='average'), 0.58, criterion='distance')
    groups_tight = split_oversized_clusters_dist(D_consensus, groups_tight, max_cluster_size=14)
else:
    groups_tight = np.ones(N_test, dtype=int)

assert np.bincount(groups_tight).max() <= 18, "Cluster size invariant violated!"
print(f'[Module 3] Formed {len(np.unique(groups_tight))} consensus clusters (max_size={np.bincount(groups_tight).max()}).')

# 5. Top-1 Gated Claude Specialist Refinement
claude_classes = ["claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5", "claude-fable-5"]
claude_idx = [LAB2I[c] for c in claude_classes if c in LAB2I]
top1 = np.argmax(LOGIT, axis=1)
claude_rows = np.where(np.isin(top1, claude_idx))[0]
if len(claude_rows) > 0 and claude_spec_logits is not None:
    c_lp = log_softmax(claude_spec_logits[claude_rows])
    c_lp_centered = c_lp - c_lp.mean(axis=1, keepdims=True)
    LOGIT[claude_rows[:, None], claude_idx] += 0.25 * c_lp_centered
    print(f'[Module 3] Applied gated Claude specialist refinement on {len(claude_rows)} rows.')

# 6. Leave-One-Out Cluster Logit Contrast (LOO-CLC) + Soft Hungarian Assignment Boost
print('[Module 3] Applying Leave-One-Out Cluster Logit Contrast (LOO-CLC) on groups_tight...')
LOGIT = log_softmax(LOGIT)
LOGIT_debiased = LOGIT.copy()
n_debiased = 0

for g in np.unique(groups_tight):
    rows = np.flatnonzero(groups_tight == g)
    m = len(rows)
    if 2 <= m <= 18:
        L_grp = LOGIT[rows]  # (m, 18)
        sum_L = L_grp.sum(axis=0, keepdims=True)
        loo_mean = (sum_L - L_grp) / float(m - 1)
        L_contrasted = log_softmax(L_grp - 0.22 * loo_mean)

        # Soft 1-to-1 Hungarian Assignment Boost within the prompt cluster
        row_ind, col_ind = linear_sum_assignment(-L_contrasted)
        L_contrasted[row_ind, col_ind] += 0.25
        LOGIT_debiased[rows] = log_softmax(L_contrasted)
        n_debiased += m

LOGIT = LOGIT_debiased
print(f'[Module 3] Completed LOO-CLC debiasing and soft Hungarian boost on {n_debiased}/{N_test} rows.')
log_ram('After Consensus Clustering & LOO-CLC')


### Step 7: Module 4 — Exact 300-Step Multiplicative Sinkhorn Balancing & Submission


In [ ]:
print('\n=== [Module 4] Exact 300-Step Multiplicative Sinkhorn Balancing ===')

def sinkhorn(P: np.ndarray, iters: int = 300) -> np.ndarray:
    """
    Exact multiplicative column scaling so every class receives an equal share (N / 18)
    of total probability mass, with rows renormalized to simplex at every step.
    """
    if len(P) < 50:
        return P
    P = np.clip(P.astype(np.float64), 1e-15, 1.0)
    c = np.ones(P.shape[1], dtype=np.float64)
    target = np.full(P.shape[1], len(P) / float(P.shape[1]), dtype=np.float64)
    for _ in range(iters):
        R = P * c
        R /= R.sum(axis=1, keepdims=True).clip(min=1e-15)
        c *= target / R.sum(axis=0).clip(min=1e-15)
    R = P * c
    return R / R.sum(axis=1, keepdims=True).clip(min=1e-15)

P_raw = softmax(LOGIT, axis=1)
if N_test >= 50:
    print(f'[Sinkhorn] Running exact 300 iterations on N={N_test} test predictions...')
    P_final = sinkhorn(P_raw, iters=300)
    print(f'[Sinkhorn] Multiplicative balancing completed: column sum range=[{P_final.sum(axis=0).min():.2f}, {P_final.sum(axis=0).max():.2f}].')
else:
    print(f'[Sinkhorn] Small test set (N={N_test} < 50); bypassing Sinkhorn.')
    P_final = P_raw

final_pred_idx = np.argmax(P_final, axis=1)
final_pred_labels = [LABELS[i] for i in final_pred_idx]

sub_df = pd.DataFrame({
    'id': df_test['id'].astype(str),
    'model': final_pred_labels
})
sub_file = working_dir / 'submission.parquet'
sub_df.to_parquet(sub_file, index=False)

print(f'\n[SUCCESS] Wrote valid submission to {sub_file} ({sub_file.stat().st_size} bytes)!')
print(f'Submission shape: {sub_df.shape}')
print(f'First 5 Predictions:\n{sub_df.head()}')
print(f'\nPredicted Model Distribution:\n{sub_df["model"].value_counts()}')
log_ram('Final Complete Notebook Execution')
